In [1]:
#%pip install pandas requests sqlalchemy pymysql scikit-learn numpy

In [90]:
%pip freeze

aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
alembic==1.18.4
annotated-doc==0.0.4
annotated-types==0.7.0
anyio==4.12.1
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.1
async-lru==2.2.0
attrs==25.4.0
babel==2.18.0
beautifulsoup4==4.14.3
bleach==6.3.0
blinker==1.9.0
boto3==1.43.6
botocore==1.43.6
cachetools==7.1.1
catboost==1.2.10
certifi==2026.2.25
cffi==2.0.0
charset-normalizer==3.4.5
click==8.3.3
cloudpickle==3.1.2
comm==0.2.3
contourpy==1.3.3
cryptography==46.0.5
cycler==0.12.1
databricks-sdk==0.106.0
debugpy==1.8.20
decorator==5.2.1
defusedxml==0.7.1
docker==7.1.0
executing==2.2.1
fastapi==0.136.1
fastjsonschema==2.21.2
Flask==3.1.3
flask-cors==6.0.2
fonttools==4.62.1
fqdn==1.5.1
frozenlist==1.8.0
gitdb==4.0.12
GitPython==3.1.50
google-auth==2.52.0
graphene==3.4.3
graphql-core==3.2.8
graphql-relay==3.2.0
graphviz==0.21
greenlet==3.3.2
gunicorn==25.3.0
h11==0.16.0
httpcore==1.0.9
httpx==0.28.1
huey==2.6.0
idna==3.11
importlib_metadata==8.7.1


In [1]:
# ============================================================
# 📦 IMPORTS
# ============================================================

import requests
import pandas as pd
import numpy as np
import sqlalchemy as sa

from sqlalchemy import text
from sqlalchemy.engine import Engine
from sklearn.model_selection import train_test_split

from datetime import datetime
from typing import List, Dict, Optional

In [42]:
# ============================================================
# ⚙️ CONFIGURACIÓN GENERAL
# ============================================================

# -----------------------------
# API CONFIG
# -----------------------------
API_BASE_URL = "http://localhost:8003"

HEALTH_ENDPOINT = f"{API_BASE_URL}/health"
BATCH_ENDPOINT = f"{API_BASE_URL}/batch"

# -----------------------------
# DATABASE CONFIG
# -----------------------------
MYSQL_USER = "mlops_user"
MYSQL_PASSWORD = "mlops_pass"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "mlops_db"

DATABASE_URL = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

# -----------------------------
# PIPELINE CONFIG
# -----------------------------
BATCH_SIZE = 1000

RAW_TABLE = "raw_diabetes_data"
CLEAN_TABLE = "clean_diabetes_data"
SPLIT_TABLE = "dataset_split"

# porcentaje splits
TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15

In [3]:
# ============================================================
# 🔌 CREAR CONEXIÓN SQLALCHEMY
# ============================================================

def create_engine_connection() -> Engine:
    """
    Crea y retorna una conexión SQLAlchemy hacia MySQL.

    Returns
    -------
    Engine
        Engine de SQLAlchemy conectado a MySQL.
    """

    engine = sa.create_engine(
        DATABASE_URL,
        pool_pre_ping=True
    )

    return engine

In [4]:
# ============================================================
# 🩺 VALIDAR DISPONIBILIDAD API
# ============================================================

def validate_api_health() -> bool:
    """
    Verifica que la API se encuentre disponible.

    Returns
    -------
    bool
        True si la API responde correctamente.
    """

    response = requests.get(HEALTH_ENDPOINT)

    if response.status_code != 200:
        raise Exception(
            f"API no disponible. Status code: {response.status_code}"
        )

    print("✅ API disponible")

    return True

In [5]:
# ============================================================
# 📥 OBTENER BATCH DE DATOS
# ============================================================

def fetch_batch(batch_size: int = 1000) -> pd.DataFrame:
    """
    Consume el endpoint batch de la API y retorna un DataFrame.

    Parameters
    ----------
    batch_size : int
        Cantidad de registros solicitados.

    Returns
    -------
    pd.DataFrame
        Datos obtenidos desde la API.
    """

    params = {
        "batch_size": batch_size
    }

    response = requests.get(
        BATCH_ENDPOINT,
        params=params
    )

    if response.status_code != 200:
        raise Exception(
            f"Error consumiendo batch API: {response.text}"
        )

    payload = response.json()

    data = payload["data"]

    df = pd.DataFrame(data)

    print(f"✅ Batch obtenido: {len(df)} registros")

    return df

In [62]:
# ============================================================
# 🧹 NORMALIZAR MISSING VALUES
# ============================================================

def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reemplaza los valores '?' por None/NaN.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame original.

    Returns
    -------
    pd.DataFrame
        DataFrame limpio.
    """

    df = df.replace("?", "missing")
    df = df.fillna("missing)

    return df

In [7]:
# ============================================================
# 🔤 NORMALIZAR NOMBRES COLUMNAS
# ============================================================

def normalize_column_names(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Normaliza nombres de columnas para SQL.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    df = df.copy()

    df.columns = [
        col.replace("-", "_")
        .replace(" ", "_")
        .lower()
        for col in df.columns
    ]

    return df

# ============================================================
# 🧾 CREAR TABLA RAW
# ============================================================

def create_raw_table(
    engine: Engine,
    sample_df: pd.DataFrame,
    drop_if_exists: bool = False
):
    """
    Crea la tabla RAW dinámicamente a partir
    de las columnas retornadas por la API.

    Parameters
    ----------
    engine : Engine

    sample_df : pd.DataFrame

    drop_if_exists : bool
        Si es True elimina la tabla antes de crearla.
    """

    with engine.begin() as conn:

        # --------------------------------------------------
        # DROP TABLE
        # --------------------------------------------------
        if drop_if_exists:

            conn.execute(
                text(f"DROP TABLE IF EXISTS {RAW_TABLE}")
            )

            print("⚠️ Tabla RAW eliminada")

        # --------------------------------------------------
        # COLUMNAS DINÁMICAS
        # --------------------------------------------------
        columns_sql = []

        for col in sample_df.columns:

            safe_col = col.replace("-", "_")

            if col == "encounter_id":
                col_type = "BIGINT"

            elif pd.api.types.is_numeric_dtype(sample_df[col]):
                col_type = "DOUBLE"

            else:
                col_type = "TEXT"

            columns_sql.append(
                f"`{safe_col}` {col_type}"
            )

        # --------------------------------------------------
        # METADATA
        # --------------------------------------------------
        metadata_cols = [
            "`load_id` VARCHAR(100)",
            "`load_timestamp` DATETIME",
            "`source` VARCHAR(100)",
            "`record_status` VARCHAR(50)"
        ]

        columns_sql.extend(metadata_cols)

        # --------------------------------------------------
        # CREATE TABLE
        # --------------------------------------------------
        create_sql = f"""
        CREATE TABLE {RAW_TABLE} (
            id BIGINT AUTO_INCREMENT PRIMARY KEY,
            {",".join(columns_sql)},
            UNIQUE (`encounter_id`)
        )
        """

        conn.execute(text(create_sql))

    print("✅ Tabla RAW creada correctamente")

In [8]:
# ============================================================
# 🧾 CREAR TABLA RAW
# ============================================================

def create_raw_table(
    engine: Engine,
    sample_df: pd.DataFrame,
    drop_if_exists: bool = False
):
    """
    Crea la tabla RAW dinámicamente.

    Parameters
    ----------
    engine : Engine

    sample_df : pd.DataFrame

    drop_if_exists : bool
        Si es True elimina la tabla antes de crearla.
    """

    with engine.begin() as conn:

        # --------------------------------------------------
        # DROP TABLE
        # --------------------------------------------------
        if drop_if_exists:

            conn.execute(
                text(f"DROP TABLE IF EXISTS {RAW_TABLE}")
            )

            print("⚠️ Tabla RAW eliminada")

        # --------------------------------------------------
        # COLUMNAS
        # --------------------------------------------------
        columns_sql = []

        for col in sample_df.columns:

            safe_col = (
                col.replace("-", "_")
                .replace(" ", "_")
                .lower()
            )

            if col == "encounter_id":
                col_type = "BIGINT"

            elif pd.api.types.is_numeric_dtype(
                sample_df[col]
            ):
                col_type = "DOUBLE"

            else:
                col_type = "TEXT"

            columns_sql.append(
                f"`{safe_col}` {col_type}"
            )

        # --------------------------------------------------
        # METADATA
        # --------------------------------------------------
        metadata_cols = [
            "`load_id` VARCHAR(100)",
            "`load_timestamp` DATETIME",
            "`source` VARCHAR(100)",
            "`record_status` VARCHAR(50)"
        ]

        columns_sql.extend(metadata_cols)

        # --------------------------------------------------
        # CREATE SQL
        # --------------------------------------------------
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {RAW_TABLE} (
            id BIGINT AUTO_INCREMENT PRIMARY KEY,
            {",".join(columns_sql)},
            UNIQUE (`encounter_id`)
        )
        """

        conn.execute(text(create_sql))

    print("✅ Tabla RAW validada")

In [9]:
# ============================================================
# 🔍 VALIDACIONES BÁSICAS
# ============================================================

def validate_raw_data(df: pd.DataFrame):
    """
    Ejecuta validaciones básicas de calidad.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a validar.
    """

    if df.empty:
        raise Exception("DataFrame vacío")

    if "encounter_id" not in df.columns:
        raise Exception("No existe encounter_id")

    duplicated = df["encounter_id"].duplicated().sum()

    if duplicated > 0:
        raise Exception(
            f"Existen {duplicated} duplicados"
        )

    print("✅ Validaciones básicas completadas")

In [10]:
# ============================================================
# 🧼 PREPROCESAMIENTO CLEAN
# ============================================================

def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ejecuta transformaciones básicas sobre los datos.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame RAW.

    Returns
    -------
    pd.DataFrame
        DataFrame transformado.
    """

    df = df.copy()

    # ----------------------------------------
    # NORMALIZAR MISSING
    # ----------------------------------------
    df = normalize_missing_values(df)

    # ----------------------------------------
    # CONVERTIR COLUMNAS NUMÉRICAS
    # ----------------------------------------
    numeric_cols = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses"
    ]

    for col in numeric_cols:

        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # ----------------------------------------
    # TARGET BINARIO
    # ----------------------------------------
    if "readmitted" in df.columns:

        df["target"] = (
            df["readmitted"]
            .apply(lambda x: 1 if x in ["<30", ">30"] else 0)
        )

    return df

In [11]:
# ============================================================
# 🎯 GENERAR SPLITS TRAIN/VAL/TEST
# ============================================================

def assign_dataset_split(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Asigna cada registro a train/validation/test.

    Si alguna clase tiene muy pocos ejemplos,
    se realiza un split sin estratificación.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    df = df.copy()

    # --------------------------------------------------------
    # VALIDAR TARGET
    # --------------------------------------------------------
    if "target" not in df.columns:
        raise Exception(
            "La columna target no existe"
        )

    # --------------------------------------------------------
    # CONTEO CLASES
    # --------------------------------------------------------
    class_counts = df["target"].value_counts()

    print("\n📊 Distribución target:")
    print(class_counts)

    # --------------------------------------------------------
    # VALIDAR SI SE PUEDE ESTRATIFICAR
    # --------------------------------------------------------
    can_stratify = class_counts.min() >= 2

    if can_stratify:

        print("✅ Split estratificado")

        train_df, temp_df = train_test_split(
            df,
            test_size=(1 - TRAIN_SIZE),
            random_state=42,
            stratify=df["target"]
        )

        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=42,
            stratify=temp_df["target"]
        )

    else:

        print(
            "⚠️ Muy pocos ejemplos por clase. "
            "Split sin estratificación."
        )

        train_df, temp_df = train_test_split(
            df,
            test_size=(1 - TRAIN_SIZE),
            random_state=42
        )

        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=42
        )

    # --------------------------------------------------------
    # ASIGNAR LABELS
    # --------------------------------------------------------
    train_df["dataset_split"] = "train"

    val_df["dataset_split"] = "validation"

    test_df["dataset_split"] = "test"

    # --------------------------------------------------------
    # CONCATENAR
    # --------------------------------------------------------
    final_df = pd.concat([
        train_df,
        val_df,
        test_df
    ])

    print("\n✅ Split completado")
    print(
        final_df["dataset_split"]
        .value_counts()
    )

    return final_df

In [12]:
def create_clean_table(
    engine,
    sample_df,
    drop_if_exists=True
):
    """
    Crea la tabla CLEAN_DATA.

    Parameters
    ----------
    engine : sqlalchemy.Engine
        Conexión a MySQL.

    sample_df : pd.DataFrame
        DataFrame ejemplo para inferir columnas.

    drop_if_exists : bool
        Si True elimina la tabla antes de crearla.
    """

    # ======================================================
    # COPIA SEGURA
    # ======================================================

    df = sample_df.copy()

    # ======================================================
    # ELIMINAR COLUMNAS TÉCNICAS DUPLICADAS
    # ======================================================

    forbidden_columns = {
        "id"
    }

    valid_columns = [
        col for col in df.columns
        if col.lower() not in forbidden_columns
    ]

    df = df[valid_columns]

    # ======================================================
    # CREAR TABLA
    # ======================================================

    with engine.begin() as conn:

        # --------------------------------------------------
        # DROP TABLE
        # --------------------------------------------------

        if drop_if_exists:
            conn.execute(
                text(
                    f"DROP TABLE IF EXISTS {CLEAN_TABLE}"
                )
            )

        # --------------------------------------------------
        # VALIDAR SI EXISTE
        # --------------------------------------------------

        table_exists = conn.execute(
            text(f"""
                SELECT COUNT(*)
                FROM information_schema.tables
                WHERE table_schema = DATABASE()
                AND table_name = '{CLEAN_TABLE}'
            """)
        ).scalar()

        if table_exists:
            print("ℹ️ Tabla CLEAN ya existe")
            return

        # --------------------------------------------------
        # SQL COLUMNAS
        # --------------------------------------------------

        sql_columns = []

        for col in df.columns:

            dtype = str(df[col].dtype).lower()

            if "int" in dtype:
                sql_type = "BIGINT"

            elif "float" in dtype:
                sql_type = "DOUBLE"

            elif "datetime" in dtype:
                sql_type = "DATETIME"

            else:
                sql_type = "TEXT"

            sql_columns.append(
                f"`{col}` {sql_type}"
            )

        columns_sql = ",\n".join(sql_columns)

        # --------------------------------------------------
        # CREATE SQL
        # --------------------------------------------------

        create_sql = f"""
        CREATE TABLE {CLEAN_TABLE} (

            id BIGINT AUTO_INCREMENT PRIMARY KEY,

            {columns_sql},

            UNIQUE (`encounter_id`)
        )
        """

        conn.execute(
            text(create_sql)
        )

    print("✅ Tabla CLEAN creada")

In [13]:
# ============================================================
# 💾 INSERTAR CLEAN DATA
# ============================================================

def insert_clean_data(
    engine,
    clean_df
):
    """
    Inserta datos en CLEAN_DATA.
    """

    df = clean_df.copy()

    # ======================================================
    # ELIMINAR ID TÉCNICO
    # ======================================================

    df = df.drop(
        columns=["id"],
        errors="ignore"
    )

    # ======================================================
    # INSERT
    # ======================================================

    df.to_sql(
        CLEAN_TABLE,
        con=engine,
        if_exists="append",
        index=False
    )

    print(
        f"✅ {len(df)} registros insertados en CLEAN"
    )

In [18]:
# ============================================================
# 📦 CARGA INCREMENTAL RAW
# ============================================================

def insert_raw_incremental(
    engine: Engine,
    df: pd.DataFrame,
    source: str = "api"
):
    """
    Inserta registros nuevos en la tabla RAW.

    Los duplicados se controlan mediante encounter_id.

    Parameters
    ----------
    engine : Engine
        Engine SQLAlchemy.

    df : pd.DataFrame
        Datos a insertar.

    source : str
        Origen de los datos.
    """

    if df.empty:
        print("⚠️ DataFrame vacío")
        return

    load_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    df["load_id"] = load_id
    df["load_timestamp"] = datetime.now()
    df["source"] = source
    df["record_status"] = "RAW"

    existing_ids_query = f"""
    SELECT encounter_id
    FROM {RAW_TABLE}
    """

    existing_df = pd.read_sql(
        existing_ids_query,
        engine
    )

    existing_ids = set(existing_df["encounter_id"].tolist())

    df = df[
        ~df["encounter_id"].isin(existing_ids)
    ]

    if df.empty:
        print("⚠️ No hay nuevos registros")
        return

    df.to_sql(
        RAW_TABLE,
        engine,
        if_exists="append",
        index=False
    )

    print(f"✅ Insertados {len(df)} registros RAW")

In [14]:
# ============================================================
# 🚀 PIPELINE COMPLETO
# ============================================================

def run_pipeline(erase_raw = True, erase_clean = True):
    """
    Ejecuta el pipeline completo de preprocesamiento.

    Flujo:
    1. Validar API
    2. Descargar batch
    3. Crear tabla RAW
    4. Validaciones
    5. Inserción incremental RAW
    6. Preprocesamiento
    7. Split train/val/test
    8. Crear tabla CLEAN
    9. Insertar CLEAN
    """

    print("=" * 60)
    print("🚀 INICIANDO PIPELINE")
    print("=" * 60)

    # ----------------------------------------
    # CONEXIÓN DB
    # ----------------------------------------
    engine = create_engine_connection()

    # ----------------------------------------
    # VALIDAR API
    # ----------------------------------------
    validate_api_health()

    # ----------------------------------------
    # FETCH DATA
    # ----------------------------------------
    raw_df = fetch_batch(
        batch_size=BATCH_SIZE
    )
    
    raw_df = normalize_column_names(raw_df)

    # ----------------------------------------
    # VALIDACIONES
    # ----------------------------------------
    validate_raw_data(raw_df)

    # ----------------------------------------
    # CREAR TABLA RAW
    # ----------------------------------------
    create_raw_table(
        engine,
        raw_df,
        drop_if_exists=erase_raw
    )

    # ----------------------------------------
    # INSERT RAW INCREMENTAL
    # ----------------------------------------
    insert_raw_incremental(
        engine,
        raw_df
    )
    
    # ----------------------------------------
    # LEER RAW HISTÓRICO
    # ----------------------------------------
    raw_historical_df = pd.read_sql(
        f"SELECT * FROM {RAW_TABLE}",
        engine
    )
    
    # ----------------------------------------
    # PREPROCESS TODO RAW
    # ----------------------------------------
    clean_df = preprocess_data(
        raw_historical_df
    )
    
    # ----------------------------------------
    # SPLITS
    # ----------------------------------------
    clean_df = assign_dataset_split(
        clean_df
    )
    
    # ----------------------------------------
    # REBUILD CLEAN
    # ----------------------------------------
    create_clean_table(
        engine,
        clean_df,
        drop_if_exists=erase_clean
    )
    
    insert_clean_data(
        engine,
        clean_df
    )

    print("=" * 60)
    print("✅ PIPELINE FINALIZADO")
    print("=" * 60)

In [19]:
def task_validate_source():
    validate_api_health()
    return True

def task_ingest_raw():
    engine = create_engine_connection()

    df = fetch_batch(BATCH_SIZE)
    df = normalize_column_names(df)

    validate_raw_data(df)

    create_raw_table(engine, df, drop_if_exists=False)
    insert_raw_incremental(engine, df)

    return len(df)

def task_validate_raw():
    engine = create_engine_connection()

    df = pd.read_sql(f"SELECT * FROM {RAW_TABLE}", engine)

    validate_raw_data(df)

    return {
        "rows": len(df),
        "duplicates": int(df["encounter_id"].duplicated().sum())
    }

def task_preprocess():
    engine = create_engine_connection()

    raw_df = pd.read_sql(f"SELECT * FROM {RAW_TABLE}", engine)

    clean_df = preprocess_data(raw_df)

    return len(clean_df)

def task_store_clean():
    engine = create_engine_connection()

    raw_df = pd.read_sql(f"SELECT * FROM {RAW_TABLE}", engine)

    clean_df = preprocess_data(raw_df)

    create_clean_table(engine, clean_df, drop_if_exists=True)
    insert_clean_data(engine, clean_df)

    return len(clean_df)

def task_split_data():
    engine = create_engine_connection()

    df = pd.read_sql(f"SELECT * FROM {CLEAN_TABLE}", engine)

    df = assign_dataset_split(df)

    # mejor práctica: guardar en misma tabla o tabla separada
    df.to_sql(
        "dataset_split",
        engine,
        if_exists="replace",
        index=False
    )

    return df["dataset_split"].value_counts().to_dict()

In [91]:
# ============================================================
# 🚀 DAG SIMULATION RUNNER
# ============================================================

def run_dag_simulation():
    """
    Simula la ejecución de un DAG de Airflow en orden secuencial.
    Cada función representa una task independiente.
    """

    print("=" * 60)
    print("🚀 INICIANDO SIMULACIÓN DEL DAG")
    print("=" * 60)

    # --------------------------------------------------------
    # TASK 1: VALIDAR FUENTE
    # --------------------------------------------------------
    print("\n🟦 TASK 1: validate_source")

    task_validate_source()
    print("✅ Fuente validada")

    # --------------------------------------------------------
    # TASK 2: INGESTA RAW
    # --------------------------------------------------------
    print("\n🟦 TASK 2: ingest_raw")

    rows = task_ingest_raw()
    print(f"✅ Ingesta completada ({rows} registros)")

    # --------------------------------------------------------
    # TASK 3: VALIDAR RAW
    # --------------------------------------------------------
    print("\n🟦 TASK 3: validate_raw")

    validation_info = task_validate_raw()
    print(f"✅ Raw validado: {validation_info}")

    # --------------------------------------------------------
    # TASK 4: PREPROCESSING
    # --------------------------------------------------------
    print("\n🟦 TASK 4: preprocess")

    n_clean = task_preprocess()
    print(f"✅ Preprocesamiento completado ({n_clean} registros)")

    # --------------------------------------------------------
    # TASK 5: STORE CLEAN
    # --------------------------------------------------------
    print("\n🟦 TASK 5: store_clean")

    n_final = task_store_clean()
    print(f"✅ CLEAN almacenado ({n_final} registros)")

    # --------------------------------------------------------
    # TASK 6: SPLIT DATASET
    # --------------------------------------------------------
    print("\n🟦 TASK 6: split_dataset")

    split_info = task_split_data()
    print(f"✅ Split completado: {split_info}")

    # --------------------------------------------------------
    # FINAL
    # --------------------------------------------------------
    print("\n" + "=" * 60)
    print("✅ DAG SIMULATION COMPLETADO EXITOSAMENTE")
    print("=" * 60)

    return {
        "ingested_rows": rows,
        "clean_rows": n_final,
        "split": split_info
    }

In [94]:
result = run_dag_simulation()

🚀 INICIANDO SIMULACIÓN DEL DAG

🟦 TASK 1: validate_source
✅ API disponible
✅ Fuente validada

🟦 TASK 2: ingest_raw
✅ Batch obtenido: 1000 registros
✅ Validaciones básicas completadas
✅ Tabla RAW validada
✅ Insertados 980 registros RAW
✅ Ingesta completada (1000 registros)

🟦 TASK 3: validate_raw
✅ Validaciones básicas completadas
✅ Raw validado: {'rows': 2971, 'duplicates': 0}

🟦 TASK 4: preprocess
✅ Preprocesamiento completado (2971 registros)

🟦 TASK 5: store_clean
✅ Tabla CLEAN creada
✅ 2971 registros insertados en CLEAN
✅ CLEAN almacenado (2971 registros)

🟦 TASK 6: split_dataset

📊 Distribución target:
target
0    1613
1    1358
Name: count, dtype: int64
✅ Split estratificado

✅ Split completado
dataset_split
train         2079
validation     446
test           446
Name: count, dtype: int64
✅ Split completado: {'train': 2079, 'validation': 446, 'test': 446}

✅ DAG SIMULATION COMPLETADO EXITOSAMENTE


In [83]:
# ============================================================
# 🔎 VALIDAR RAW
# ============================================================

engine = create_engine_connection()

raw_preview = pd.read_sql(
    f"SELECT * FROM {RAW_TABLE}",
    engine
)

raw_preview

,id,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,...,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesmed,readmitted,load_id,load_timestamp,source,record_status
0,1,158838726,89041761.0,AfricanAmerican,Female,[40-50),?,Emergency,Discharged to home,Emergency Room,...,No,No,No,Ch,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
1,2,352494608,125529566.0,Caucasian,Male,[70-80),?,Elective,Discharged to home,Physician Referral,...,No,No,No,Ch,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
2,3,84454926,21393603.0,Caucasian,Male,[50-60),?,Urgent,Discharged/transferred to home with home healt...,Transfer from a hospital,...,No,No,No,Ch,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
3,4,12455328,343548.0,Caucasian,Male,[20-30),?,Urgent,Discharged to home,Physician Referral,...,No,No,No,No,No,>30,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
4,5,155582202,96790491.0,Caucasian,Female,[60-70),?,Not Available,Discharged to home,Physician Referral,...,No,No,No,No,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
5,6,141551820,23103918.0,AfricanAmerican,Male,[60-70),?,Emergency,Discharged to home,Emergency Room,...,No,No,No,No,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
6,7,241984380,40881933.0,Caucasian,Female,[50-60),?,Elective,Discharged to home,Physician Referral,...,No,No,No,Ch,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
7,8,121827708,84060045.0,Caucasian,Female,[60-70),[100-125),Urgent,Discharged/transferred to home with home healt...,Physician Referral,...,No,No,No,Ch,Yes,>30,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
8,9,44443848,1691541.0,AfricanAmerican,Female,[60-70),?,Urgent,Discharged to home,Physician Referral,...,No,No,No,No,Yes,NO,batch_20260510_191810,2026-05-10 19:18:10,api,RAW
9,10,44805048,9944991.0,Caucasian,Female,[60-70),?,Urgent,NULL,Physician Referral,...,No,No,No,No,No,>30,batch_20260510_191810,2026-05-10 19:18:10,api,RAW


In [88]:
# ============================================================
# 🔎 VALIDAR CLEAN
# ============================================================

clean_preview = pd.read_sql(
    f"SELECT * FROM {CLEAN_TABLE}",
    engine
)

clean_preview

,id,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,...,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesmed,readmitted,load_id,load_timestamp,source,record_status,target
0,1,70342710,212625.0,AfricanAmerican,Male,[70-80),missing,Emergency,Discharged/transferred to SNF,Emergency Room,...,No,No,Ch,Yes,NO,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,0
1,2,140726772,25109082.0,Caucasian,Female,[90-100),missing,Emergency,Discharged/transferred to SNF,Emergency Room,...,No,No,No,No,NO,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,0
2,3,10862544,5153868.0,Caucasian,Female,[50-60),missing,Emergency,Discharged to home,Emergency Room,...,No,No,No,Yes,>30,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,1
3,4,364433576,127204700.0,Caucasian,Male,[50-60),missing,Emergency,Discharged/transferred to home with home healt...,Emergency Room,...,No,No,Ch,Yes,>30,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,1
4,5,116469372,2995812.0,AfricanAmerican,Female,[80-90),missing,Emergency,Discharged/transferred to SNF,Emergency Room,...,No,No,Ch,Yes,>30,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,1984,71356710,1374129.0,Caucasian,Female,[60-70),missing,Urgent,Discharged to home,Emergency Room,...,No,No,No,Yes,>30,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,1
1984,1985,434280836,105943590.0,Caucasian,Female,[70-80),[50-75),Emergency,Expired,Transfer from a Skilled Nursing Facility (SNF),...,No,No,No,No,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0
1985,1986,267743874,32397219.0,Caucasian,Female,[60-70),missing,Urgent,Discharged/transferred to home with home healt...,Physician Referral,...,No,No,Ch,Yes,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0
1986,1987,443816024,106392411.0,Caucasian,Female,[70-80),missing,Elective,Discharged/transferred to home with home healt...,Physician Referral,...,No,No,Ch,Yes,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0


In [89]:
clean_preview = pd.read_sql(
    f"SELECT * FROM {SPLIT_TABLE}",
    engine
)

clean_preview

,id,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,...,metformin_pioglitazone,change,diabetesmed,readmitted,load_id,load_timestamp,source,record_status,target,dataset_split
0,1595,126415728,24353820.0,AfricanAmerican,Male,[50-60),missing,Emergency,Discharged to home,Emergency Room,...,No,Ch,Yes,<30,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,1,train
1,765,166868214,80966520.0,AfricanAmerican,Female,[10-20),missing,Emergency,Discharged to home,Emergency Room,...,No,No,Yes,>30,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,1,train
2,883,255202962,97605432.0,Caucasian,Female,[80-90),missing,Urgent,Discharged/transferred to home with home healt...,Emergency Room,...,No,Ch,Yes,NO,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,0,train
3,125,80887050,2083437.0,Caucasian,Male,[60-70),missing,Elective,Discharged/transferred to another rehab fac in...,Physician Referral,...,No,Ch,Yes,<30,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,1,train
4,1919,195418548,67114368.0,missing,Female,[70-80),missing,Not Available,Discharged/transferred to home with home healt...,Physician Referral,...,No,No,No,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,769,291110238,81244071.0,Caucasian,Female,[50-60),[75-100),Emergency,Discharged/transferred to SNF,Transfer from a Skilled Nursing Facility (SNF),...,No,No,No,NO,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,0,test
1984,1141,225856734,42530283.0,Caucasian,Male,[70-80),missing,Elective,Discharged to home,Physician Referral,...,No,No,Yes,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0,test
1985,264,185896998,41416416.0,Other,Male,[40-50),missing,Emergency,Discharged to home,Emergency Room,...,No,No,No,NO,batch_20260510_192303,2026-05-10 19:23:04,api,RAW,0,test
1986,1894,270027144,96875208.0,AfricanAmerican,Female,[70-80),missing,NULL,Discharged to home,Emergency Room,...,No,No,Yes,NO,batch_20260510_202552,2026-05-10 20:25:52,api,RAW,0,test


In [40]:
clean_preview.columns

Index(['id', 'encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'a1cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide_metformin', 'glipizide_metformin',
       'glimepiride_pioglitazone', 'metformin_rosiglitazone',
       'metformin_pioglitazone', 'change', 'diabetesmed', 'readmitted',
       'load_id', 'load_timestamp', 'source', 'record_status', 'target'],
      dtype=

In [25]:
# ============================================================
# ▶️ EJECUCIÓN
# ============================================================

run_pipeline(False, True)

🚀 INICIANDO PIPELINE
✅ API disponible
✅ Batch obtenido: 50 registros
✅ Validaciones básicas completadas
✅ Tabla RAW validada
✅ Insertados 50 registros RAW

📊 Distribución target:
target
0    107
1     93
Name: count, dtype: int64
✅ Split estratificado

✅ Split completado
dataset_split
train         139
test           31
validation     30
Name: count, dtype: int64
✅ Tabla CLEAN creada
✅ 200 registros insertados en CLEAN
✅ PIPELINE FINALIZADO


In [134]:
clean_preview["dataset_split"].value_counts()

dataset_split
train         452
test           98
validation     97
Name: count, dtype: int64

In [81]:
import pandas as pd
from io import StringIO

# Leer archivo completo
with open("diabetes+130-us+hospitals+for+years+1999-2008/IDS_mapping.csv", "r", encoding="utf-8") as f:
    content = f.read()

# Separar bloques por filas vacías
blocks = content.strip().split("\n,\n")

tables = {}

for block in blocks:
    df = pd.read_csv(StringIO(block))
    
    # El nombre de la tabla lo tomamos de la primera columna
    table_name = df.columns[0]
    
    tables[table_name] = df

# Acceder a cada tabla
admission_type = tables["admission_type_id"]
discharge = tables["discharge_disposition_id"]
admission_source = tables["admission_source_id"]

print(admission_type.head())
print(discharge.head())
print(admission_source.head())

   admission_type_id    description
0                  1      Emergency
1                  2         Urgent
2                  3       Elective
3                  4        Newborn
4                  5  Not Available
   discharge_disposition_id                                        description
0                         1                                 Discharged to home
1                         2  Discharged/transferred to another short term h...
2                         3                      Discharged/transferred to SNF
3                         4                      Discharged/transferred to ICF
4                         5  Discharged/transferred to another type of inpa...
   admission_source_id                                      description
0                    1                               Physician Referral
1                    2                                  Clinic Referral
2                    3                                     HMO Referral
3                    4